# Cat orbit: Floquet multipliers up to and through its fold (19/10/2026)

Referee report v2, point 9: *does the cat itself ever lose stability?* Same model and conventions as
`Hill_method_corrected_cat_lobe_stability.ipynb`: $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$,
$\kappa_b = \pi/10.4$ (Julia convention, $C = \mathrm{diag}(0, \kappa_b/\omega_b)$), $\tau = \omega_a t$, drive at harmonic 2, orbit period $T = 2\pi$.

**Why the sweep fails at high $\varepsilon_p$.** The cat orbit does not exist above a fold, a saddle-node of limit cycles.
At $\varepsilon_d = 10.12\,g_2$ the fold is at $\varepsilon_p^\ast \simeq 0.786$ ($g_2/2\pi \simeq 97.2$ MHz). There the cat annihilates with a saddle orbit.
Above $\varepsilon_p^\ast$ the harmonic-balance equations have no cat solution, so Powell cannot converge (or jumps to another branch).
Natural-parameter continuation cannot turn a fold. Pseudo-arclength continuation, in $(z, \varepsilon_p)$, follows the branch through it.

Changes with respect to the previous notebook (results unchanged where both apply):
1. **Exact AFT over one period.** $N$ samples, `endpoint=False`, $\Gamma^+ = (2/N)\Gamma^T$, instead of `pinv` on 50 periods. It is faster, so a whole sweep takes seconds.
2. **No globals.** The parameters ($\beta'$, $\varepsilon_d'$) live in the `HB` object.
3. **Pseudo-arclength continuation** through the fold.
4. **Cross-check of the Hill multipliers** (genuine exponents selected as before) against a direct monodromy matrix, from the variational equation integrated around the HB orbit.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root
from scipy.linalg import block_diag, eig as scipy_eig
from scipy.special import jv
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *

OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/"

In [ ]:
#### Caracterization of the circuit (same as the Hill notebooks) #####
w_a = 25.338776456203686
w_b = 2*w_a
phi_a = 0.11
phi_b = 0.204
E_J = 37.12*2*np.pi*1.5
kappa_b = np.pi/10.4   # 1/ns, Julia convention (phi_b'' = ... - 2 kappa_b phi_b')

def g2_of(eps_p):
    return np.sin(eps_p)*E_J*phi_a**2*phi_b

def g2_J1_of(eps_p):
    # former definition, only used to continue the hard-coded seed
    return jv(1, eps_p)*E_J*phi_a**2*phi_b

def MHz(g2):
    return g2/2/np.pi*1e3

n = 2
M = np.diag([1.0, 0.25])
C = np.diag([0.0, kappa_b/w_b])
K = np.eye(2)

## Harmonic balance with an exact one-period AFT
Same ordering as before: $z = [\mathbf c_0, \mathbf s_1, \mathbf c_1, \dots]$, each block $(a, b)$; $\mathbf X = \mathbf c_0/\sqrt2 + \sum_k \mathbf s_k\sin k\tau + \mathbf c_k\cos k\tau$.

In [ ]:
def build_L(N_H):
    dim = n*(2*N_H + 1)
    L = np.zeros((dim, dim))
    L[:n, :n] = K
    for k in range(1, N_H + 1):
        A, Bk = K - k**2*M, k*C
        s, c = 2*k - 1, 2*k
        L[s*n:(s+1)*n, s*n:(s+1)*n] = A
        L[s*n:(s+1)*n, c*n:(c+1)*n] = -Bk
        L[c*n:(c+1)*n, s*n:(s+1)*n] = Bk
        L[c*n:(c+1)*n, c*n:(c+1)*n] = A
    return L

def build_Gamma(t, N_H):
    k = np.arange(1, N_H + 1)
    Phi = np.empty((t.size, 2*N_H + 1))
    Phi[:, 0] = 1/np.sqrt(2)
    Phi[:, 1::2] = np.sin(np.outer(t, k))
    Phi[:, 2::2] = np.cos(np.outer(t, k))
    return np.kron(Phi, np.eye(n))          # time-major stacking


class HB:
    # One period sampled uniformly (endpoint=False): Gamma^+ = (2/N) Gamma^T exactly (with the 1/sqrt2 DC convention)
    def __init__(self, N_H, N=None):
        self.N_H, self.N = N_H, N or 64*N_H
        self.L = build_L(N_H)
        self.Gamma = build_Gamma(np.linspace(0, 2*np.pi, self.N, endpoint=False), N_H)
        self.Gamma_pinv = (2/self.N)*self.Gamma.T

    def set_params(self, eps_p, eps_d):
        self.B = 2*E_J*np.sin(eps_p)*np.array([phi_a**2/w_a, phi_b**2/w_b])    # (beta_a', beta_b')
        self.eps_d_p = eps_d*phi_b/w_b
        self.b_ext = np.zeros(n*(2*self.N_H + 1))
        self.b_ext[2*2*n + 1] = -np.sqrt(2)*self.eps_d_p                        # c_2 block, mode b

    def X(self, z):
        return (self.Gamma @ z).reshape(self.N, n).T

    def residual(self, z):
        X = self.X(z)
        Fnl = -np.sqrt(2)*self.B[:, None]*(np.cos(np.sqrt(2)*(X[0] + X[1])) - 1)[None, :]
        return self.L @ z + self.Gamma_pinv @ Fnl.T.reshape(-1) - self.b_ext

    def jacobian(self, z):
        X = self.X(z)
        s = np.sin(np.sqrt(2)*(X[0] + X[1]))
        JG = np.einsum('j,ab,jbm->jam', 2*s, np.outer(self.B, [1, 1]),
                       self.Gamma.reshape(self.N, n, -1)).reshape(n*self.N, -1)
        return self.L + self.Gamma_pinv @ JG

    def solve(self, z0, tol=1e-12):
        sol = root(self.residual, z0, jac=self.jacobian, method="hybr", tol=tol,
                   options={"maxfev": 100*(len(z0) + 1)})
        if not sol.success:
            raise RuntimeError(f"Powell hybrid did not converge: {sol.message}")
        return sol.x


def orbit(z, N_H, t):
    # x and dx/dtau of both modes at times t
    k = np.arange(1, N_H + 1)
    out = []
    for zz in (z[::2], z[1::2]):
        s, c = zz[1::2], zz[2::2]
        x = zz[0]/np.sqrt(2) + np.sin(np.outer(t, k)) @ s + np.cos(np.outer(t, k)) @ c
        v = np.cos(np.outer(t, k)) @ (k*s) - np.sin(np.outer(t, k)) @ (k*c)
        out.append((x, v))
    return out

def mean_abs_aR(z, N_H):
    # mean |a_R| over the orbit, a = (x + i p)/(sqrt2 phi_a) with p = dx/dtau
    t = np.linspace(0, 2*np.pi, 400, endpoint=False)
    (x_a, v_a), _ = orbit(z, N_H, t)
    return np.mean(np.abs(x_a + 1j*v_a))/(np.sqrt(2)*phi_a)

def strobe(z, N_H, tau=0.0):
    # state [x_a, x_b, v_a, v_b] on the orbit at time tau
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, np.array([tau]))
    return np.array([x_a[0], x_b[0], v_a[0], v_b[0]])

## Floquet multipliers: Hill (genuine exponents) and monodromy
`select_genuine` is copied from `Hill_method_corrected_cat_lobe_stability.ipynb`.
The monodromy matrix integrates $M\delta\ddot{\mathbf X} + C\delta\dot{\mathbf X} + (K + \partial_X f_{nl}(\mathbf X(\tau)))\delta\mathbf X = 0$ over one period, which is an independent check.

In [ ]:
def complex_harmonics(U, N_H, n):
    A = np.zeros((2*N_H + 1, n, U.shape[1]), dtype=complex)
    A[N_H] = U[:n, :] / np.sqrt(2)
    for k in range(1, N_H + 1):
        s = U[(2*k - 1)*n:(2*k)*n, :]
        c = U[(2*k)*n:(2*k + 1)*n, :]
        A[N_H + k] = (c - 1j*s) / 2
        A[N_H - k] = (c + 1j*s) / 2
    return A

def harmonic_centroid(A, N_H):
    w = np.sum(np.abs(A)**2, axis=1)
    k = np.arange(-N_H, N_H + 1)[:, None]
    return np.sum(k*w, axis=0) / np.sum(w, axis=0)

def shifted_overlap(a, b, m, N_H):
    if m > 0:
        a_s, b_s = a[m:], b[:-m]
    elif m < 0:
        a_s, b_s = a[:m], b[-m:]
    else:
        a_s, b_s = a, b
    return np.abs(np.vdot(a_s, b_s)) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-300)

def select_genuine(eigvals, eigvecs, N_H, n, tol=1e-3, overlap_min=0.9):
    U = eigvecs[eigvecs.shape[0]//2:, :]
    A = complex_harmonics(U, N_H, n)
    centroid = harmonic_centroid(A, N_H)
    order = np.lexsort((np.abs(np.imag(eigvals)), np.abs(centroid)))
    chosen = []
    for j in order:
        lam = eigvals[j]
        if not np.isfinite(lam):
            continue
        same_family = False
        for c in chosen:
            d = lam - eigvals[c]
            m = int(np.round(np.imag(d)))
            if (m != 0 and abs(np.real(d)) < tol and abs(np.imag(d) - m) < tol
                    and shifted_overlap(A[:, :, c], A[:, :, j], m, N_H) > overlap_min):
                same_family = True
                break
        if not same_family:
            chosen.append(j)
        if len(chosen) == 2*n:
            break
    chosen = np.array(chosen)
    lam = eigvals[chosen]
    lam_folded = lam - 1j*np.round(np.imag(lam))
    lam_folded = np.where(np.abs(np.imag(lam_folded) + 0.5) < 1e-6, lam_folded + 1j, lam_folded)
    return lam_folded, eigvecs[:, chosen]


def floquet_hill(hb, z):
    # Hill quadratic eigenproblem lambda^2 Delta2 + lambda Delta1 + h(z) = 0, then the 2n genuine exponents -> multipliers exp(2 pi lambda)
    N_H, d = hb.N_H, (2*hb.N_H + 1)*n
    Delta_1 = block_diag(*([C]*(2*N_H + 1)))
    for k in range(1, N_H + 1):
        i, j = 2*k - 1, 2*k
        Delta_1[i*n:(i+1)*n, j*n:(j+1)*n] = -2*k*M
        Delta_1[j*n:(j+1)*n, i*n:(i+1)*n] = 2*k*M
    Delta_2 = block_diag(*([M]*(2*N_H + 1)))
    B_1 = np.block([[Delta_1, hb.jacobian(z)], [-np.eye(d), np.zeros((d, d))]])
    B_2 = -np.block([[Delta_2, np.zeros((d, d))], [np.zeros((d, d)), np.eye(d)]])
    lam, _ = select_genuine(*scipy_eig(B_1, B_2), N_H, n)
    return np.exp(2*np.pi*lam)


def floquet_monodromy(hb, z):
    Minv, base = np.linalg.inv(M), np.outer(hb.B, [1, 1])
    def rhs(t, Y):
        (x_a, _), (x_b, _) = orbit(z, hb.N_H, np.atleast_1d(t))
        A = np.zeros((4, 4))
        A[:2, 2:] = np.eye(2)
        A[2:, :2] = -Minv @ (K + 2*np.sin(np.sqrt(2)*(x_a[0] + x_b[0]))*base)
        A[2:, 2:] = -Minv @ C
        return (A @ Y.reshape(4, 4)).reshape(-1)
    sol = solve_ivp(rhs, (0, 2*np.pi), np.eye(4).reshape(-1), method="DOP853", rtol=1e-11, atol=1e-13)
    return np.linalg.eigvals(sol.y[:, -1].reshape(4, 4))

## Seed and natural continuation in $\varepsilon_p$ ($\varepsilon_d = 10.12\,g_2$)
The hard-coded cat seed was converged with the former $g_2$ (J1). It is first continued in $\varepsilon_d$ at $\varepsilon_p = 0.5$, as in the previous notebook.

In [ ]:
N_H = 16
ratio = 10.12
hb = HB(N_H)

z_seed = np.array([-2.09501523e-02, -3.60273363e-02,  3.26246220e-01, -3.28386996e-17,
       -6.78344780e-03,  1.43507376e-17, -1.72514305e-04, -3.13979476e-03,
       -4.23320965e-03, -7.46552651e-02, -8.01435361e-04, -8.82665625e-03,
        5.05426053e-05,  4.29993690e-04,  7.79954440e-06,  6.01698525e-05,
        1.00817384e-04,  8.67341068e-04])   # cat
z_seed = np.pad(z_seed, (0, n*(2*N_H + 1) - len(z_seed)))

def cat_seed(hb, ratio, eps_p=0.5):
    z = np.pad(z_seed[:n*(2*hb.N_H + 1)], (0, max(0, n*(2*hb.N_H + 1) - len(z_seed))))
    for eps_d in np.linspace(10.12*g2_J1_of(eps_p), 10.12*g2_of(eps_p), 25):
        hb.set_params(eps_p, eps_d)
        z = hb.solve(z)
    for r in np.linspace(10.12, ratio, 1 + int(np.ceil(abs(ratio - 10.12)/0.05)))[1:]:
        hb.set_params(eps_p, r*g2_of(eps_p))
        z = hb.solve(z)
    return z

z = cat_seed(hb, ratio)
eps_p = 0.5
step = 0.005
while True:
    try:
        hb.set_params(eps_p + step, ratio*g2_of(eps_p + step))
        z = hb.solve(z)
        eps_p += step
    except RuntimeError as e:
        print(f"Failed at eps_p = {eps_p + step:.3f}: {e}")
        break
hb.set_params(eps_p, ratio*g2_of(eps_p))
print(f"last converged eps_p = {eps_p:.3f} (g2/2pi = {MHz(g2_of(eps_p)):.1f} MHz), |mu| = {np.round(np.abs(floquet_hill(hb, z)), 4)}")
z_last, eps_p_last = z.copy(), eps_p

The last converged point is still strongly stable ($|\mu| \le 0.97$), and yet the next step fails. This is the signature of a fold:
the branch turns back, and nothing is left to converge to on the far side.

## Pseudo-arclength continuation through the fold
Unknowns $y = (z, \varepsilon_p)$. Predictor along the tangent (the null vector of $[\partial_z R, \partial_{\varepsilon_p} R]$), corrector with the arclength constraint $t\cdot(y - y_{\rm prev}) = \Delta s$.

In [ ]:
def continue_arclength(hb, z, eps_p, ratio, ds=0.004, n_steps=60, stop=None):
    def R(y):
        hb.set_params(y[-1], ratio*g2_of(y[-1]))
        return hb.residual(y[:-1])
    def dR(y):
        hb.set_params(y[-1], ratio*g2_of(y[-1]))
        Jz, r0 = hb.jacobian(y[:-1]), hb.residual(y[:-1])
        h = 1e-7
        hb.set_params(y[-1] + h, ratio*g2_of(y[-1] + h))
        return np.column_stack([Jz, (hb.residual(y[:-1]) - r0)/h])
    def tangent(y, t_prev=None):
        t = np.linalg.svd(dR(y))[2][-1]
        if (t_prev is not None and t @ t_prev < 0) or (t_prev is None and t[-1] < 0):
            t = -t
        return t
    y = np.concatenate([z, [eps_p]])
    t = tangent(y)
    branch = [y.copy()]
    while len(branch) <= n_steps:
        sol = root(lambda w: np.concatenate([R(w), [t @ (w - y) - ds]]), y + ds*t,
                   jac=lambda w: np.vstack([dR(w), t]), method="hybr", tol=1e-12)
        if not sol.success:
            ds /= 2
            if ds < 1e-6:
                break
            continue
        y = sol.x
        t = tangent(y, t)
        branch.append(y.copy())
        if stop is not None and stop(y, t):
            break
    return np.array(branch)

branch = continue_arclength(hb, z_last, eps_p_last, ratio, ds=0.002, n_steps=2800)   # 2800: up to the re-stabilised satellite (~114 MHz)
eps_branch = branch[:, -1]

# turning points of eps_p along the branch. The cat fold is the FIRST one: on a long branch eps_p
# comes back up (satellite) and exceeds it, so np.argmax(eps_branch) would point at the end of the branch
d_eps = np.diff(eps_branch)
i_turn = np.where(np.sign(d_eps[1:]) != np.sign(d_eps[:-1]))[0] + 1

def turning_eps(i):
    c = np.polyfit([-1, 0, 1], eps_branch[i - 1:i + 2], 2)
    return np.polyval(c, -c[1]/(2*c[0]))

i_fold = i_turn[0]
eps_p_fold = turning_eps(i_fold)
print(f"fold at eps_p* = {eps_p_fold:.4f}, g2*/2pi = {MHz(g2_of(eps_p_fold)):.1f} MHz (eps_d = {ratio} g2)")
for i in i_turn[1:]:
    print(f"next turning point ({'max' if d_eps[i - 1] > 0 else 'min'} of eps_p) at eps_p = {turning_eps(i):.4f}, g2/2pi = {MHz(g2_of(turning_eps(i))):.2f} MHz")

In [ ]:
# multipliers along the branch: Hill, and the monodromy on two points as a check
mu_branch = []
for y in branch:
    hb.set_params(y[-1], ratio*g2_of(y[-1]))
    mu_branch.append(floquet_hill(hb, y[:-1]))
mu_branch = np.array(mu_branch)
abs_a_branch = np.array([mean_abs_aR(y[:-1], N_H) for y in branch])

for i in (0, len(branch) - 1):
    y = branch[i]
    hb.set_params(y[-1], ratio*g2_of(y[-1]))
    mu_h = floquet_hill(hb, y[:-1])
    mu_m = floquet_monodromy(hb, y[:-1])
    diff = max(np.min(np.abs(mu_m - m)) for m in mu_h)
    print(f"eps_p = {y[-1]:.4f}  Hill |mu| = {np.round(np.sort(np.abs(mu_h)), 4)}   monodromy |mu| = {np.round(np.sort(np.abs(mu_m)), 4)}   max mismatch {diff:.1e}")
print("Liouville: prod |mu| =", np.round(np.prod(np.abs(mu_branch), axis=1)[[0, -1]], 6), " expected", np.round(np.exp(-2*np.pi*np.trace(np.linalg.solve(M, C))), 6))

In [ ]:
max(eps_branch)

In [ ]:
# lower (stable) part of the cat branch from eps_p = 0.3, to draw the whole curve
hb_lo = HB(N_H)
z = cat_seed(hb_lo, ratio)
lo = []
for e in np.arange(0.5, 0.29, -0.01):
    hb_lo.set_params(e, ratio*g2_of(e)); z = hb_lo.solve(z)
    lo.append((e, mean_abs_aR(z, N_H), np.max(np.abs(floquet_hill(hb_lo, z)))))
z = cat_seed(hb_lo, ratio)
for e in np.arange(0.51, eps_p_last, 0.01):
    hb_lo.set_params(e, ratio*g2_of(e)); z = hb_lo.solve(z)
    lo.append((e, mean_abs_aR(z, N_H), np.max(np.abs(floquet_hill(hb_lo, z)))))
lo = np.array(sorted(lo))

stable = np.max(np.abs(mu_branch), axis=1) < 1
fig, axes = plt.subplots(1, 2, figsize=(2*figsize[0], figsize[1]))
ax = axes[0]
ax.plot(MHz(g2_of(lo[:, 0])), lo[:, 1]**2, color=Perso_palette_float[2])
ax.plot(MHz(g2_of(eps_branch)), np.where(stable, abs_a_branch**2, np.nan), color=Perso_palette_float[2], label="stable")
ax.plot(MHz(g2_of(eps_branch)), np.where(stable, np.nan, abs_a_branch**2), "--", color=Perso_palette_float[4], label="unstable")
ax.axvline(MHz(g2_of(eps_p_fold)), ls=":", color="k", lw=0.8)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)")
ax.set_ylabel(r"$\langle|a_R|\rangle^2$")
ax.legend(fontsize=6)

ax = axes[1]
x = MHz(g2_of(lo[:, 0]))
ax.plot(x, lo[:, 2], color=Perso_palette_float[2])
for j in range(2*n):
    ax.plot(MHz(g2_of(eps_branch[stable])), np.abs(mu_branch[stable, j]), ".", ms=1.5, color=Perso_palette_float[2])
    ax.plot(MHz(g2_of(eps_branch[~stable])), np.abs(mu_branch[~stable, j]), ".", ms=1.5, color=Perso_palette_float[4])
ax.axhline(1, ls="--", color="r", lw=0.8)
ax.axvline(MHz(g2_of(eps_p_fold)), ls=":", color="k", lw=0.8)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)")
ax.set_ylabel(r"$|\mu|$")
plt.tight_layout()
plt.savefig(FIG_DIR + "cat_fold_floquet_19_10_2026.pdf")
plt.show()

In [ ]:
# multipliers in the complex plane near the fold: the slow complex pair meets on the real axis, then one multiplier leaves through +1
plt.figure(figsize=figsize_car)
th = np.linspace(0, 2*np.pi, 200)
plt.plot(np.cos(th), np.sin(th), "k-", lw=0.6)
sc = plt.scatter(np.real(mu_branch).ravel(), np.imag(mu_branch).ravel(),
                 c=np.repeat(np.arange(len(branch)), 2*n), s=2, cmap="viridis")
plt.colorbar(sc, label="arclength step")
plt.xlim(0.6, 1.4); plt.ylim(-0.4, 0.4)
plt.gca().set_aspect("equal")
plt.xlabel(r"Re $\mu$"); plt.ylabel(r"Im $\mu$")
plt.tight_layout()
plt.show()

### The 4 multipliers along the whole branch
Natural continuation from $\varepsilon_p = 0.5$ up to the last converged point, then the arclength branch through the fold onto the saddle orbit (dashed).
Within each row, the multipliers are ordered so that every line follows one multiplier: the slow pair (largest real parts), then the fast pair.
The third panel shows the product of each pair. If it stays constant, each pair behaves as a weakly damped Hamiltonian pair, $\mu_1\mu_2 = e^{-2\delta T}$ in place of $1$.

In [ ]:
hb_mu = HB(N_H)
z = cat_seed(hb_mu, ratio)
eps_nat = np.arange(0.5, eps_p_last + 1e-9, 0.005)
mu_nat = []
for e in eps_nat:
    hb_mu.set_params(e, ratio*g2_of(e))
    z = hb_mu.solve(z)
    mu_nat.append(floquet_hill(hb_mu, z))
mu_all = np.vstack([np.array(mu_nat), mu_branch[1:]])
g2_all = MHz(g2_of(np.concatenate([eps_nat, eps_branch[1:]])))
saddle = np.concatenate([np.zeros(len(eps_nat), bool), np.max(np.abs(mu_branch[1:]), axis=1) > 1])   # unstable points

def order_row(mu):
    # slow pair = two largest real parts; each pair ordered by Im (desc), then Re (desc) once it is real
    mu = mu[np.argsort(-np.real(mu))]
    key = lambda m: (-np.round(np.imag(m), 9), -np.real(m))
    return np.array(sorted(mu[:2], key=key) + sorted(mu[2:], key=key))
mu_all = np.array([order_row(m) for m in mu_all])

# where the slow pair reaches the real axis, compared with the fold
i_col = np.argmax(np.abs(np.imag(mu_all[:, 0])) < 1e-9)
print(f"slow pair meets the real axis at g2/2pi ~ {g2_all[i_col]:.3f} MHz, |mu| = {np.abs(mu_all[i_col - 1, 0]):.4f}; fold at {MHz(g2_of(eps_p_fold)):.3f} MHz")
prod_slow, prod_fast = np.real(mu_all[:, 0]*mu_all[:, 1]), np.real(mu_all[:, 2]*mu_all[:, 3])
print(f"slow pair product: {prod_slow.min():.5f} .. {prod_slow.max():.5f},  fast pair product: {prod_fast.min():.5f} .. {prod_fast.max():.5f}")

colors = [Perso_palette_float[2], Perso_palette_float[2], Perso_palette_float[4], Perso_palette_float[4]]
fig, axes = plt.subplots(1, 3, figsize=(3*figsize[0], 1.15*figsize[0]))
th = np.linspace(0, 2*np.pi, 300)
for ax, lims in ((axes[0], (-1.15, 1.35, -1.15, 1.15)), (axes[1], (0.7, 1.32, -0.2, 0.2))):
    ax.plot(np.cos(th), np.sin(th), "k-", lw=0.6)
    ax.plot(np.sqrt(prod_slow[0])*np.cos(th), np.sqrt(prod_slow[0])*np.sin(th), ":", color="gray", lw=0.6)
    for j in range(4):
        ax.plot(np.real(np.where(saddle, np.nan, mu_all[:, j])), np.imag(np.where(saddle, np.nan, mu_all[:, j])), "-", color=colors[j], lw=1)
        ax.plot(np.real(np.where(saddle, mu_all[:, j], np.nan)), np.imag(np.where(saddle, mu_all[:, j], np.nan)), "--", color=colors[j], lw=1)
        ax.plot(np.real(mu_all[0, j]), np.imag(mu_all[0, j]), "o", color=colors[j], ms=3)
        ax.plot(np.real(mu_all[-1, j]), np.imag(mu_all[-1, j]), "s", mfc="none", color=colors[j], ms=3)
    ax.plot(np.real(mu_all[i_fold + len(eps_nat) - 1]), np.imag(mu_all[i_fold + len(eps_nat) - 1]), "kx", ms=4)
    ax.axhline(0, color="k", lw=0.3)
    ax.set_xlim(lims[:2]); ax.set_ylim(lims[2:]); ax.set_aspect("equal")
    ax.set_xlabel(r"Re $\mu$"); ax.set_ylabel(r"Im $\mu$")
axes[0].set_title("o: $g_2/2\pi$ = %.0f MHz, x: fold, □: end of the branch, dashed: unstable" % g2_all[0], fontsize=5)
axes[1].set_title("zoom near +1 (dotted: $|\\mu| = \\sqrt{\\mu_1\\mu_2}$ of the slow pair)", fontsize=5)
ax = axes[2]
step = np.arange(len(g2_all))
ax.plot(step, prod_slow, color=colors[0], label=r"slow pair $\mu_1\mu_2$")
ax.plot(step, prod_fast, color=colors[2], label=r"fast pair $\mu_3\mu_4$")
ax.axvline(len(eps_nat) - 1 + i_fold, ls=":", color="k", lw=0.8)
ax.set_xlabel("continuation step (fold: dotted)"); ax.set_ylabel("product of the pair")
ax.legend(fontsize=5)
plt.tight_layout()
plt.savefig(FIG_DIR + "cat_fold_multipliers_complex_plane_19_10_2026.pdf")
plt.show()

**Reading.** The fast pair (buffer-like, more damped) never meets the slow pair. It only rotates, from $\arg\mu \approx \pm0.27\pi$ to $\pm0.73\pi$.
The slow pair (memory-like) slides along a circle of radius $\sqrt{\mu_1\mu_2} \approx 0.972$ towards $+1$ and meets **its own conjugate** on the real axis.
It then splits into two real multipliers, and one of them crosses $+1$ at the fold. The collision and the fold coincide to $\Delta\varepsilon_p \sim 10^{-6}$ ($\sim$ kHz in $g_2$), because $\mathrm{Im}\,\mu \propto \sqrt{\varepsilon_p^\ast - \varepsilon_p}$.

This is the Hamiltonian **saddle-centre** (tangent) bifurcation, with weak loss added. Without loss, the pair $e^{\pm i\vartheta}$ on the unit circle would meet at $+1$ and split into a reciprocal pair $(\mu, 1/\mu)$.
Loss shrinks the circle to radius $e^{-\delta T}$. The collision becomes a focus→node change of the rotating-frame fixed point, which is not a bifurcation, and the passage through $+1$ becomes an ordinary saddle-node.
It is **not** a Hamiltonian–Hopf (Krein) collision. That would need two different pairs to meet off the real axis and leave as a complex quartet, which with loss gives a Neimark–Sacker.
The pair products are not exactly constant. Damping moves between the hybridised memory and buffer pairs (about 1%), while the total stays fixed by Liouville.

### Following the saddle branch further (`n_steps = 2500`): it ends on the satellite
At $\varepsilon_d = 10.12\,g_2$, $N_H = 16$, with stability checked against the monodromy:

| $g_2/2\pi$ | event | orbit |
|---|---|---|
| 97.24 MHz | fold (max of $\varepsilon_p$), $\mu \to +1$ | cat → saddle |
| 40.36 MHz | fold (min of $\varepsilon_p$), $\mu \to +1$ | saddle → stable large orbit |
| 75.2–75.3 MHz | $\mu \to -1$: period doubling | unstable |
| 103.6–103.8 MHz | $\mu \to -1$ again: reverse period doubling | stable again, up to $g_2^{\max}$ ($\varepsilon_p = \pi/2$) |

The phase of the first memory harmonic rotates along the saddle, from $\pm\pi/2$ (cat) to $\approx\pi$ (real axis).
The stable orbit after the lower fold is therefore the **satellite**, and its period doubling at 75.3 MHz is the one found in `Hill_method_corrected.ipynb`.
So the satellite is born at $g_2 \approx 40$ MHz in a saddle-node with the same saddle orbit that destroys the cat at 97 MHz. The cat and the satellite are the two stable ends of a single S-shaped branch.
The period-1 satellite does cross the cat-fold threshold, and it is linearly stable again above 103.7 MHz.
A time integration from the HB orbit, kicked by $10^{-4}$, relaxes back onto it (distance $< 10^{-7}$ after $200/\kappa_b$) at 108, 114, 127 and 135 MHz, so it is an attractor there.
It coexists with the chaotic set of the Lyapunov maps; its basin was not measured.

## Fold line in the $(g_2, \varepsilon_d/2g_2)$ plane, compared with the cat-start Lyapunov map
For each $\varepsilon_d/g_2$: continue the seed in the ratio at $\varepsilon_p = 0.5$, step $\varepsilon_p$ up, then use arclength until $\varepsilon_p$ turns back.
The scan is capped at $\varepsilon_p = \pi/2$, the maximum of $g_2$. $N_H = 12$ is enough here, since the tail of the spectrum is below $10^{-9}$.

In [ ]:
N_H_f = 12
hb_f = HB(N_H_f)
ratio_list = np.round(np.arange(2.0, 13.51, 0.25), 3)

def fold_of_ratio(z, ratio):
    e = 0.5
    while e + 0.01 <= np.pi/2:
        hb_f.set_params(e + 0.01, ratio*g2_of(e + 0.01))
        try:
            z_new = hb_f.solve(z)
        except RuntimeError:
            break
        if abs(z_new[2]) < 0.5*abs(z[2]):     # jumped off the cat branch
            break
        z, e = z_new, e + 0.01
    else:
        return np.nan                          # no fold below g2_max
    br = continue_arclength(hb_f, z, e, ratio, ds=0.002, n_steps=200,
                            stop=lambda y, t: t[-1] < 0 or y[-1] > np.pi/2)
    eb = br[:, -1]
    if eb[-1] >= np.pi/2 or len(eb) < 3:
        return np.nan
    i = np.argmax(eb)
    i = min(max(i, 1), len(eb) - 2)
    c = np.polyfit([-1, 0, 1], eb[i - 1:i + 2], 2)
    return np.polyval(c, -c[1]/(2*c[0]))

# seeds: continuation in eps_d/g2 at eps_p = 0.5, both ways from 10.12
seeds = {}
z10 = cat_seed(hb_f, 10.12)
for part in (ratio_list[ratio_list >= 10.12], ratio_list[ratio_list < 10.12][::-1]):
    z, r_prev = z10.copy(), 10.12
    for r in part:
        for rr in np.linspace(r_prev, r, 6)[1:]:
            hb_f.set_params(0.5, rr*g2_of(0.5)); z = hb_f.solve(z)
        seeds[r], r_prev = z.copy(), r

eps_p_fold_list = np.array([fold_of_ratio(seeds[r], r) for r in ratio_list])
g2_fold_MHz = MHz(g2_of(eps_p_fold_list))
for r, e, g in zip(ratio_list[::4], eps_p_fold_list[::4], g2_fold_MHz[::4]):
    print(f"eps_d/g2 = {r:5.2f}   eps_p* = {e:.4f}   g2*/2pi = {g:6.1f} MHz")

with open(OUT_DIR + "hill_orbits/cat_fold_line_19_10_2026.json", "w") as f:
    json.dump({"description": "Fold (saddle-node of limit cycles) of the cat orbit, Hill_method_cat_fold.ipynb. NaN: no fold below g2_max (eps_p = pi/2).",
               "g2_definition": "E_J sin(eps_p) phi_a^2 phi_b", "kappa_b": kappa_b, "N_H": N_H_f,
               "eps_d_over_g2": ratio_list.tolist(),
               "eps_p_fold": [None if np.isnan(e) else float(e) for e in eps_p_fold_list],
               "g2_fold_MHz": [None if np.isnan(g) else float(g) for g in g2_fold_MHz]}, f)

In [ ]:
L_cat = np.array(json.load(open(OUT_DIR + "../archive/data/data_lyapunov_cat_start_12_10_2026.json")))   # (eps_p, eps_d/g2), sign of the exponent
L_origin = np.array(json.load(open(OUT_DIR + "lyapunov_maps/data_lyapunov_12_10_2026.json")))
g2_map = MHz(np.array(json.load(open(OUT_DIR + "../archive/data/g2_list_cat_start_12_10_2026.json"))))
alpha_sq_map = np.array(json.load(open(OUT_DIR + "../archive/data/alpha_sq_list_cat_start_12_10_2026.json")))

def onset(L):
    # smallest g2 with a positive exponent, for each eps_d/2g2
    return np.array([g2_map[np.argmax(col == 1)] if (col == 1).any() else np.nan for col in L.T])
g2_onset_cat, g2_onset_origin = onset(L_cat), onset(L_origin)

fig, ax = plt.subplots(figsize=figsize)
ax.pcolormesh(g2_map, alpha_sq_map, L_cat.T, cmap="RdBu_r", shading="nearest", vmin=-1.5, vmax=1.5, rasterized=True)
ax.plot(g2_onset_origin, alpha_sq_map, color="gold", lw=1, label="chaos onset, origin start")
ax.plot(g2_fold_MHz, ratio_list/2, color="k", lw=1.2, label="fold of the cat orbit (Hill)")
ax.set_xlim(g2_map[0], g2_map[-1]); ax.set_ylim(alpha_sq_map[0], alpha_sq_map[-1])
ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\varepsilon_d/2g_2$")
ax.legend(fontsize=5, loc="lower left")
plt.tight_layout()
plt.savefig(FIG_DIR + "heatmap_lyapunov_cat_start_fold_19_10_2026.pdf")
plt.show()

g2_fold_on_map = np.interp(2*alpha_sq_map, ratio_list, g2_fold_MHz)
ok = np.isfinite(g2_onset_cat) & np.isfinite(g2_fold_on_map)
print("g2_fold / g2_onset(cat start): mean %.3f, min %.3f, max %.3f"
      % tuple(f(g2_fold_on_map[ok]/g2_onset_cat[ok]) for f in (np.mean, np.min, np.max)))
print("g2_fold - g2_onset(cat start) (MHz): min %.1f, max %.1f" % (np.min((g2_fold_on_map - g2_onset_cat)[ok]), np.max((g2_fold_on_map - g2_onset_cat)[ok])))

## Why the cat start is chaotic before the fold: the basin shrinks
The cat-start map starts from $a(0) = i|\alpha|$, $b(0) = 0$, which is not on the cat orbit.
Here the full EOMs, the same model as the HB, are integrated from $a(0) = i r|\alpha|$, $b(0) = 0$, for $120/\kappa_b$ (code convention).
`C` means the trajectory ends within $10^{-3}$ of one of the two cats, at the stroboscopic time $\tau = 2\pi k$. The second cat is the same lab-frame orbit shifted by half a period.
All initial conditions at one $\varepsilon_p$ are integrated together, in one vectorised ODE.
Two extra starts separate the two parts of the mismatch: "a on orbit, b = 0", and "a = i|α|, b on orbit".

In [ ]:
def full_rhs_batch(eps_p, eps_d):
    beta = 2*E_J*np.sin(eps_p)*np.array([phi_a**2/w_a, phi_b**2/w_b])
    eps_d_p = eps_d*phi_b/w_b
    def f(t, Y):
        x_a, x_b, v_a, v_b = Y.reshape(4, -1)
        g = np.cos(np.sqrt(2)*(x_a + x_b)) - 1
        acc_a = -x_a + np.sqrt(2)*beta[0]*g
        acc_b = 4*(-x_b - np.sqrt(2)*eps_d_p*np.cos(2*t) + np.sqrt(2)*beta[1]*g - C[1, 1]*v_b)
        return np.concatenate([v_a, v_b, acc_a, acc_b])
    return f

r_list = np.round(np.arange(0.6, 1.41, 0.05), 2)
eps_basin = [0.60, 0.65, 0.68, 0.70, 0.71, 0.72, 0.74, 0.76, 0.78]
alpha = np.sqrt(ratio/2)
T_end = 2*np.pi*int(round(120*w_a/kappa_b/(2*np.pi)))
hb_b = HB(N_H)
z = cat_seed(hb_b, ratio)
e_cur = 0.5
print("r =", " ".join(f"{r:.2f}" for r in r_list))
for e in eps_basin:
    for ee in np.linspace(e_cur, e, 1 + int(np.ceil((e - e_cur)/0.005)))[1:]:
        hb_b.set_params(ee, ratio*g2_of(ee)); z = hb_b.solve(z)
    e_cur = e
    y_plus, y_minus = strobe(z, N_H, 0.0), strobe(z, N_H, np.pi)
    Y0 = [np.array([0, 0, np.sqrt(2)*phi_a*alpha*r, 0]) for r in r_list]
    Y0 += [np.array([y_plus[0], 0, y_plus[2], 0]), np.array([0, y_plus[1], np.sqrt(2)*phi_a*alpha, y_plus[3]])]
    Y0 = np.array(Y0).T
    sol = solve_ivp(full_rhs_batch(e, ratio*g2_of(e)), (0, T_end), Y0.reshape(-1), method="DOP853",
                    rtol=1e-10, atol=1e-12, t_eval=[T_end])
    YT = sol.y[:, -1].reshape(4, -1)
    d = np.minimum(np.linalg.norm(YT - y_plus[:, None], axis=0), np.linalg.norm(YT - y_minus[:, None], axis=0))
    lands = np.where(d < 1e-3, "C", ".")
    print(f"eps_p = {e:.2f} (g2/2pi = {MHz(g2_of(e)):5.1f} MHz): {''.join(lands[:len(r_list)])}   "
          f"a on orbit, b = 0: {lands[-2]}   a = i|alpha|, b on orbit: {lands[-1]}")

## Analytical fold (article App. "Analytical derivation of the harmonic balance")
The single-harmonic cat of App. "Cat states", $J_2(2|a_0|) = \varepsilon_d'/(2\beta_b')$, has no fold here.
Its right-hand side is $r\varphi_a^2/4 \approx 0.03$ for every $\varepsilon_p$, far below $\max J_2 \simeq 0.49$.
The fold comes from the corrections of the satellite Ansatz ($a_0, a_{\pm1}, b_0, b_{-1}, b_2$) taken on the **cat family $\sigma = -1$**.
In the article's notation, the corrections are the static junction phase $C$, the total $2\omega_a$ amplitude $\bar y = y + z$ and the $3\omega_a$ sideband $w = u x$.

**Quadratic order, $\kappa_b' \to 0$.** Eq. (app_sat_poly) with $\sigma = -1$ gives, as for the satellites, $C = -\bar y(1+u)$ and $\eta u^2 + (1+\eta)u - \eta = 0$ with $\eta = 2\beta_b'\bar y$. Then
$$x^2(1+2u) = 2(1+u)\bar y^2 + A,\qquad \bar y(1+u) = 2\beta_\Sigma\Bigl[\Bigl(1+\tfrac{(1+u)^2}{2}\Bigr)\bar y^2 + (1+u^2)x^2\Bigr],\qquad A \equiv \frac{\varepsilon_d'}{\beta_b'} = \varphi_a^2|\alpha|^2 .$$
The only difference from the satellites is the sign of $A$.

**Leading order ($u = 0$).** $x^2 = A + 2\bar y^2$ and $\bar y = 2\beta_\Sigma(A + \tfrac72\bar y^2)$, i.e. $7\beta_\Sigma\bar y^2 - \bar y + 2\beta_\Sigma A = 0$:
$$\bar y_\mp = \frac{1 \mp \sqrt{1 - 56\,\beta_\Sigma^2 A}}{14\,\beta_\Sigma}.$$
$\bar y_-$ is the cat; it tends to the bare cat when $\beta_\Sigma \to 0$. $\bar y_+$ is the saddle orbit found by the arclength continuation.
They merge at $56\,\beta_\Sigma^2 A = 1$. The cat exists only while the static phase $C$, which detunes the memory, can be cancelled by the parametric $2\omega_a$ component, $\bar y \simeq -C$.
But $\bar y$ itself feeds $\langle X^2\rangle$, and hence $C$, so the self-consistency loses its real root.
With $\beta_\Sigma^2 A = \bigl[g_2|\alpha|(2\varphi_a^2+\varphi_b^2)/(\omega_a\varphi_a\varphi_b)\bigr]^2$:
$$g_2^\ast\,|\alpha| = \sqrt{s^\ast}\,\frac{\omega_a\varphi_a\varphi_b}{2\varphi_a^2+\varphi_b^2},\qquad s^\ast = \frac1{56}\ \text{(leading order)} .$$
This has two consequences. The fold line is $g_2^\ast \propto 1/|\alpha| = \sqrt{2g_2/\varepsilon_d}$, and $\kappa_b$ drops out.

**First order in $u$.** With $u \simeq 2\rho q$, $q = \beta_\Sigma\bar y$ and $\rho = \beta_b'/\beta_\Sigma = \varphi_b^2/(2\varphi_a^2+\varphi_b^2)$, the scalar equation becomes $s \equiv \beta_\Sigma^2 A = q/2 - (7/2 - 3\rho)q^2 - 12\rho q^3$.
Its maximum is $s^\ast = (1 + 30\rho/49)/56$. The $3\omega_a$ sideband is small, $u \approx 0.1$ at the fold, but it moves $g_2^\ast$ by about 17% because the fold sits at a maximum of $s(q)$.

In [ ]:
from scipy.special import jv as J

rho = phi_b**2/(2*phi_a**2 + phi_b**2)
K_fold = w_a*phi_a*phi_b/(2*phi_a**2 + phi_b**2)          # g2* |alpha| = sqrt(s*) K_fold

def u_of(eta):
    return (np.sqrt((1 + eta)**2 + 4*eta**2) - (1 + eta))/(2*eta)

def s_of_q(q):
    # quadratic-order scalar equation solved for s = beta_Sigma^2 A, all orders in u (kappa_b' -> 0)
    u = u_of(2*rho*q)
    return (q*(1 + u)/2 - (1 + (1 + u)**2/2)*q**2)*(1 + 2*u)/(1 + u**2) - 2*(1 + u)*q**2

q_grid = np.linspace(1e-4, 0.3, 300001)
s_star = {"leading order, 1/56": 1/56,
          "first order in u": (1 + 30*rho/49)/56,
          "quadratic order, all u": np.max(s_of_q(q_grid))}
for name, s in s_star.items():
    print(f"{name:24s}  56 s* = {56*s:.4f}   g2*|alpha|/2pi = {MHz(np.sqrt(s)*K_fold):6.1f} MHz   at eps_d = 10.12 g2: {MHz(np.sqrt(s)*K_fold)/np.sqrt(10.12/2):5.1f} MHz")
print(f"Hill (N_H = 12): g2*|alpha|/2pi = {np.nanmean(g2_fold_MHz*np.sqrt(ratio_list/2)):.1f} MHz (mean over eps_d/g2)")

**Reduced system with the full cosine.** Eqs. (app_sat_bessel) and (app_RN) with $\sigma = -1$ and $\kappa_b'$ kept. The unknowns are $(x, \bar y, w, C, y)$, continued in $\varepsilon_p$ by pseudo-arclength.
The article's $\kappa_b' = \kappa_b/(2\omega_b)$ uses the GKSL $\kappa_b$, which is twice the code's, so numerically it is `kappa_b/w_b` with the code's `kappa_b`, as in `C`.

In [ ]:
m_, p_ = np.meshgrid(np.arange(-8, 9), np.arange(-8, 9), indexing="ij")

def R_N(N, x, yb, w, C, quad):
    if quad:   # cos X - 1 ~ -X^2/2
        return {-1: -x*(C + yb) - yb*w, 0: 1 - (C**2/2 + x**2 + yb**2 + w**2),
                -2: -(C*yb + x**2/2 + x*w), -3: -(C*w + x*yb)}[N]
    return np.sum(J(N - 2*m_ - 3*p_, 2*x)*J(m_, 2*yb)*J(p_, 2*w)*np.cos(C + (N - m_ - 2*p_)*np.pi/2))

def reduced_residual(v, eps_p, r, quad=False, kappa_p=kappa_b/w_b, sigma=-1):
    x, yb, w, C, y = v
    s = np.sin(eps_p)
    beta_a, beta_b = 2*E_J*s*phi_a**2/w_a, 2*E_J*s*phi_b**2/w_b
    eps_d_p = r*g2_of(eps_p)*phi_b/w_b
    gamma = 1 + beta_b/(beta_a*(1 + kappa_p**2)) if not quad else 1 + beta_b/beta_a
    eps_eff = np.sqrt(max(eps_d_p**2 - 4*kappa_p**2*y**2, 0.0))
    R = lambda N: R_N(N, x, yb, w, C, quad)
    return np.array([R(-1),
                     C - 2*gamma*beta_a*(R(0) - 1),
                     2*beta_b*R(-2) - sigma*eps_eff,
                     w + 2*beta_b*R(-3),
                     y - yb - sigma*beta_a*eps_eff/(2*beta_b)])

def fold_reduced(r, quad=False, kappa_p=kappa_b/w_b, eps_p0=0.3, ds=0.003):
    G = lambda Y: reduced_residual(Y[:-1], Y[-1], r, quad, kappa_p)
    def dG(Y, h=1e-8):
        g0 = G(Y)
        return np.column_stack([(G(Y + h*e) - g0)/h for e in np.eye(6)])
    x0 = np.sqrt(r*phi_a**2/2)                                  # bare cat, x = phi_a |alpha|
    sol = root(lambda v: reduced_residual(v, eps_p0, r, quad, kappa_p), [x0, 0, 0, 0, 0], tol=1e-13)
    Y = np.append(sol.x, eps_p0)
    t = np.linalg.svd(dG(Y))[2][-1]
    t = t if t[-1] > 0 else -t
    eps_hist = [eps_p0]
    while True:
        s = root(lambda Z: np.append(G(Z), t @ (Z - Y) - ds), Y + ds*t, tol=1e-13)
        if not s.success:
            ds /= 2
            if ds < 1e-7:
                return np.nan
            continue
        Y = s.x
        t_new = np.linalg.svd(dG(Y))[2][-1]
        t = t_new if t_new @ t > 0 else -t_new
        eps_hist.append(Y[-1])
        if t[-1] < 0:
            c = np.polyfit([-1, 0, 1], eps_hist[-3:], 2)
            return np.polyval(c, -c[1]/(2*c[0]))
        if Y[-1] > np.pi/2:
            return np.nan

for kf in (0.25, 1, 4):
    print(f"kappa_b x {kf:4}: fold of the reduced (full-cosine) system at eps_d = 10.12 g2: {MHz(g2_of(fold_reduced(10.12, kappa_p=kf*kappa_b/w_b))):.2f} MHz")

In [ ]:
r_red = ratio_list[np.isfinite(g2_fold_MHz)]
g2_red_full = np.array([MHz(g2_of(fold_reduced(r))) for r in r_red])
g2_red_quad = np.array([MHz(g2_of(fold_reduced(r, quad=True))) for r in r_red])
print("max |reduced full - Hill| =", np.round(np.nanmax(np.abs(g2_red_full - g2_fold_MHz[np.isfinite(g2_fold_MHz)])), 2), "MHz")

alpha_ = np.linspace(1, np.sqrt(13.5/2), 200)
fig, ax = plt.subplots(figsize=figsize)
ax.plot(g2_fold_MHz, ratio_list/2, color="k", lw=2.5, alpha=0.3, label="Hill (converged)")
ax.plot(g2_red_full, r_red/2, color=Perso_palette_float[2], lw=1, label="reduced system, full cosine")
ax.plot(g2_red_quad, r_red/2, ":", color=Perso_palette_float[4], lw=1, label="reduced system, quadratic")
ax.plot(MHz(np.sqrt(s_star["first order in u"])*K_fold)/alpha_, alpha_**2, "--", color=Perso_palette_float[5], lw=1,
        label=r"$\sqrt{(1+30\rho/49)/56}\,K/|\alpha|$")
ax.plot(MHz(np.sqrt(1/56)*K_fold)/alpha_, alpha_**2, "-.", color=Perso_palette_float[3], lw=1, label=r"$K/(\sqrt{56}\,|\alpha|)$")
ax.set_xlim(60, 140); ax.set_ylim(2.5, 6.75)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\varepsilon_d/2g_2$")
ax.legend(fontsize=5)
plt.tight_layout()
plt.savefig(FIG_DIR + "cat_fold_analytic_19_10_2026.pdf")
plt.show()

## Gallery of limit cycles along the branch ($\varepsilon_d = 10.12\,g_2$)
The frame is the rotating frame of `limit_cycles.ipynb`: $a_R = i\,a\,e^{i\tau}$ and $b_R = b\,e^{2i\tau}$, with $a = (x_a + i p_a)/(\sqrt2\varphi_a)$, $b = (x_b + i p_b)/(\sqrt2\varphi_b)$, $p_a = dx_a/d\tau$ and $p_b = dx_b/d(2\tau)$.
Each period-1 orbit is drawn together with its partner $a \to -a$ (lighter). The buffer orbit is shared by both.
The thin line joins the orbit centres $\langle a_R\rangle$, $\langle b_R\rangle$ along the whole S-shaped branch: cat, fold at 97.2 MHz, saddle, fold at 40.4 MHz, then satellite.
Three period-2 orbits (period $4\pi$ in $\tau$) belong to **two different families**:
- **The period-doubled satellite**, the orbit of `Hill_method_doubling_corrected.ipynb`. It is born from the satellite at 75.3 MHz, where its odd harmonics vanish, and is stable only up to the Neimark–Sacker point at 76.75 MHz.
  - 76.0 MHz (black, solid): time integration from the satellite plus a $10^{-4}$ kick, $300/\kappa_b$ of transient, then 2 periods.
  - 100 MHz (black, dotted): harmonic balance over two periods, continued from the seed of that notebook. Here it is strongly unstable, with $\max|\mu| \approx 2.7$ (complex pair). A time integration never lands on it.
  It continues to 128 MHz and stays unstable, with $|\mu|$ up to 4.4. The odd harmonics stay small (memory odd/even $\approx$ 0.065/0.44): the orbit is a slightly doubled satellite.
- **The $\omega_a/2$ orbit** (light blue, 103.5 MHz, time integration as above). It is a separate period-2 branch dominated by the **odd** harmonics (0.54/0.11), so the memory oscillates mainly at $\omega_a/2$ (a subharmonic of the drive at $\omega_d/4$). In the rotating frame it therefore draws a large loop around the origin.
  Converted to harmonic balance and continued, it exists from about 100 to 119 MHz. It is stable on about 101.2–106.8 MHz, apart from a narrow window just below 101.2 MHz, and loses stability through a complex pair at 106.8 MHz. Its distance to the doubled satellite at the same $g_2$ is $|\Delta z| = 1.48$.
  It is the attractor reached from the satellite just below the reverse period doubling at 103.7 MHz, and it is the likely origin of the regular window of the satellite-start map at about 102–111 MHz.

In [ ]:
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)

def rotating_frame(x_a, v_a, x_b, v_b, t):
    a_R = 1j*(x_a + 1j*v_a)/(np.sqrt(2)*phi_a)*np.exp(1j*t)
    b_R = (x_b + 1j*v_b/2)/(np.sqrt(2)*phi_b)*np.exp(2j*t)
    return a_R, b_R

t_orb = np.linspace(0, 2*np.pi, 300)
def hb_orbit(z):
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, t_orb)
    return rotating_frame(x_a, v_a, x_b, v_b, t_orb)

def eps_of_MHz(g):
    return np.arcsin(g*2*np.pi*1e-3/(E_J*phi_a**2*phi_b))

# --- cats: natural continuation from eps_p = 0.5, down and up ---
hb_lc = HB(N_H)
def walk(z, e_from, e_to):
    for e in np.linspace(e_from, e_to, 1 + int(np.ceil(abs(e_to - e_from)/0.005)))[1:]:
        hb_lc.set_params(e, ratio*g2_of(e))
        z = hb_lc.solve(z)
    return z
z_half = cat_seed(hb_lc, ratio)
orbits = []                                   # (kind, g2 in MHz, a_R, b_R)
for targets in ((50, 30), (80, 95)):
    z, e = z_half, 0.5
    for g in targets:
        z = walk(z, e, eps_of_MHz(g)); e = eps_of_MHz(g)
        orbits.append(("cat", g, *hb_orbit(z)))

# --- saddle and satellite: points of the arclength branch ---
max_mu = np.max(np.abs(mu_branch), axis=1)
i_low = i_turn[1]                             # lower fold (birth of the satellite)
g2_branch = MHz(g2_of(eps_branch))
def pick(k_lo, k_hi, g):
    return k_lo + np.argmin(np.abs(g2_branch[k_lo:k_hi] - g))
for kind, (k_lo, k_hi), targets in (("saddle", (i_fold + 1, i_low), (90, 70, 50)),
                                    ("satellite, stable", (i_low, len(branch)), (45, 60, 72))):# 110, 125)),
                                    #("satellite, unstable (PD)", (i_low, len(branch)), (85, 100))):
    for g in targets:
        k = pick(k_lo, k_hi, g)
        if abs(g2_branch[k] - g) > 1:
            continue                              # branch too short to reach this g2
        orbits.append((kind, g2_branch[k], *hb_orbit(branch[k, :-1])))
        print(f"{kind:25s} g2/2pi = {g2_branch[k]:6.1f} MHz  max|mu| = {max_mu[k]:.3f}")

# --- period-2 orbits: time integration ---
T_tr = 2*np.pi*int(round(300*w_a/kappa_b/(2*np.pi)))
t2 = np.linspace(0, 4*np.pi, 600)
for g, kind in ((76.0, "period 2: doubled satellite, stable"),):#, (103.5, r"period 2: $\omega_a/2$ orbit, stable")):
    k = pick(i_low, len(branch), g)
    if abs(g2_branch[k] - g) > 1:
        continue
    e = eps_branch[k]
    f = full_rhs_batch(e, ratio*g2_of(e))
    s = solve_ivp(f, (0, T_tr), strobe(branch[k, :-1], N_H) + 1e-4, method="DOP853", rtol=1e-10, atol=1e-12)
    s = solve_ivp(f, (T_tr, T_tr + t2[-1]), s.y[:, -1], t_eval=T_tr + t2, method="DOP853", rtol=1e-10, atol=1e-12)
    x_a, x_b, v_a, v_b = s.y
    orbits.append((kind, g2_branch[k], *rotating_frame(x_a, v_a, x_b, v_b, t2)))
    print(f"{kind:40s} g2/2pi = {g2_branch[k]:6.1f} MHz  |x(T+4pi) - x(T)| = {np.linalg.norm(s.y[:, -1] - s.y[:, 0]):.1e}")

# --- the period-doubled satellite of Hill_method_doubling_corrected.ipynb at 100 MHz (HB over 2 periods) ---
# tau' = tau/2 (period 2 pi in tau'): M' = M/4, C' = C/2, drive at harmonic 4. Same seed as that notebook (eps_p = 1.2, J1 -> sin).
class HB2(HB):
    def __init__(self, N_H, N=None):
        super().__init__(N_H, N)
        M2, C2 = M/4, C/2
        for k in range(1, N_H + 1):
            A, Bk = K - k**2*M2, k*C2
            s_, c_ = 2*k - 1, 2*k
            self.L[s_*n:(s_+1)*n, s_*n:(s_+1)*n] = A
            self.L[s_*n:(s_+1)*n, c_*n:(c_+1)*n] = -Bk
            self.L[c_*n:(c_+1)*n, s_*n:(s_+1)*n] = Bk
            self.L[c_*n:(c_+1)*n, c_*n:(c_+1)*n] = A
    def set_params(self, eps_p, eps_d):
        super().set_params(eps_p, eps_d)
        self.b_ext[:] = 0
        self.b_ext[2*4*n + 1] = -np.sqrt(2)*self.eps_d_p          # c_4 block, mode b

def monodromy_p2(hb2, z):
    # variational equation over the full period 4 pi (in tau)
    Minv, base = np.linalg.inv(M), np.outer(hb2.B, [1, 1])
    def rhs(t, Y):
        (x_a, _), (x_b, _) = orbit(z, hb2.N_H, np.atleast_1d(t)/2)
        A = np.zeros((4, 4)); A[:2, 2:] = np.eye(2)
        A[2:, :2] = -Minv @ (K + 2*np.sin(np.sqrt(2)*(x_a[0] + x_b[0]))*base); A[2:, 2:] = -Minv @ C
        return (A @ Y.reshape(4, 4)).reshape(-1)
    sol = solve_ivp(rhs, (0, 4*np.pi), np.eye(4).reshape(-1), method="DOP853", rtol=1e-10, atol=1e-12)
    return np.linalg.eigvals(sol.y[:, -1].reshape(4, 4))

N_H2 = 24
hb2 = HB2(N_H2)
z2 = np.pad(np.array([-8.67480681e-02, -1.49178000e-01, -1.55487349e-04, -3.92476650e-06,
         4.80123540e-02,  6.60523141e-02, -3.00220194e-02,  3.07073460e-16,
         3.31363194e-01, -1.72222278e-16, -3.88737312e-03,  1.57262610e-02,
         3.35330370e-02, -1.65080648e-01,  6.93437379e-04, -5.59462961e-02,
        -4.67601081e-03,  3.00083819e-01,  4.52017869e-05,  9.32575051e-04,
        -4.88538089e-04, -7.81645147e-03, -1.26778042e-03, -1.46772441e-02,
         4.61868963e-03,  5.06227939e-02,  2.68514538e-04,  2.61445320e-03,
        -1.01232565e-03, -9.46918403e-03, -5.89963411e-04, -5.17649744e-03,
         1.52354566e-03,  1.30588333e-02]), (0, n*(2*N_H2 + 1) - 34))
for eps_d in np.linspace(10.12*g2_J1_of(1.2), 10.12*g2_of(1.2), 25):
    hb2.set_params(1.2, eps_d); z2 = hb2.solve(z2)


targets_p2 = (125, 94, 75.5)   # descending: continuation from eps_p = 1.2 (128 MHz)
e = 1.2
for g in targets_p2:
    e_to = eps_of_MHz(g)
    for e_new in np.linspace(e, e_to, 1 + int(np.ceil((e - e_to)/0.005)))[1:]:
        hb2.set_params(e_new, ratio*g2_of(e_new)); z2 = hb2.solve(z2)
    e = e_to
    mu2 = monodromy_p2(hb2, z2)
    za2 = z2[::2]; amp2 = np.abs(za2[1::2] + 1j*za2[2::2])
    kind = "period 2: doubled satellite, " + ("stable" if np.abs(mu2).max() < 1 else "unstable (NS)")
    print(f"{kind:42s} g2/2pi = {MHz(g2_of(e)):6.1f} MHz  max|mu| = {np.abs(mu2).max():.3f}  odd/even harmonics of x_a: {amp2[0]:.3f}/{amp2[1]:.3f}")
    (x_a, v_a), (x_b, v_b) = orbit(z2, N_H2, t2/2)
    orbits.append((kind, MHz(g2_of(e)), *rotating_frame(x_a, v_a/2, x_b, v_b/2, t2)))


# --- centres of the orbits along the branch (cat part from the natural continuation, then arclength) ---
z, e, spine = z_half, 0.5, []
for e_new in np.arange(0.5, eps_of_MHz(30) - 1e-9, -0.01):
    z = walk(z, e, e_new); e = e_new
    a_R, b_R = hb_orbit(z); spine.append((MHz(g2_of(e)), a_R.mean(), b_R.mean(), True))
spine = spine[::-1]
z, e = z_half, 0.5
for e_new in np.arange(0.51, eps_p_last, 0.01):
    z = walk(z, e, e_new); e = e_new
    a_R, b_R = hb_orbit(z); spine.append((MHz(g2_of(e)), a_R.mean(), b_R.mean(), True))
for k in range(0, len(branch), 5):
    a_R, b_R = hb_orbit(branch[k, :-1]); spine.append((g2_branch[k], a_R.mean(), b_R.mean(), max_mu[k] < 1))
sp_g2 = np.array([s[0] for s in spine]); sp_a = np.array([s[1] for s in spine])
sp_b = np.array([s[2] for s in spine]); sp_st = np.array([s[3] for s in spine])

In [ ]:
style = {"cat": dict(color=Perso_palette_float[2], ls="-"),
         "saddle": dict(color=Perso_palette_float[4], ls="--"),
         "satellite, stable": dict(color=Perso_palette_float[5], ls="-"),
         "satellite, unstable (PD)": dict(color=Perso_palette_float[5], ls=":"),
         "period 2: doubled satellite, stable": dict(color="k", ls="-"),
         "period 2: doubled satellite, unstable (NS)": dict(color="k", ls=":"),
         r"period 2: $\omega_a/2$ orbit, stable": dict(color=Perso_palette_float[3], ls="-")}

fig = plt.figure(figsize=(2*figsize_car[0], 2*figsize_car[1]))
lims = {2: 9.5, 3: 5.0}
for n_ax, (title, which, sp) in enumerate((("memory", 2, sp_a), ("buffer", 3, sp_b))):
    lab = "a" if which == 2 else "b"
    signs = (1, -1) if which == 2 else (1,)                  # partner a -> -a (the buffer orbit is shared)
    # 3d: complex plane x g2
    ax = fig.add_subplot(2, 2, n_ax + 1, projection="3d")
    for o in orbits:
        kind, g, w = o[0], o[1], o[which]
        for sign in signs:
            ax.plot(sign*np.real(w), sign*np.imag(w), g, lw=0.8, **style[kind])
            if kind == "cat":                                  # the cat orbit is almost a point
                ax.scatter(sign*np.real(w.mean()), sign*np.imag(w.mean()), g, s=4, color=style[kind]["color"])
    for sign in signs:
        ax.plot(sign*np.real(sp), sign*np.imag(sp), sp_g2, color="gray", lw=0.5)
    ax.set_xlim(-lims[which], lims[which]); ax.set_ylim(-lims[which], lims[which])
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis._axinfo["grid"].update(linewidth=0.3, color="0.85")
        axis.set_pane_color((1, 1, 1, 0))
    ax.set_xlabel(rf"Re ${lab}_R$", fontsize=6, labelpad=-10); ax.set_ylabel(rf"Im ${lab}_R$", fontsize=6, labelpad=-10)
    ax.zaxis.set_rotate_label(False)
    ax.set_zlabel(r"$g_2/2\pi$ (MHz)", fontsize=6, rotation=90, labelpad=-8)
    ax.tick_params(labelsize=4, pad=-3)
    ax.view_init(elev=18, azim=-55)
    ax.set_box_aspect(None, zoom=0.85)
    ax.set_title(title, fontsize=7)
    # top view: projection on the complex plane
    ax2 = fig.add_subplot(2, 2, n_ax + 3)
    for o in orbits:
        kind, w = o[0], o[which]
        for sign in signs:
            ax2.plot(sign*np.real(w), sign*np.imag(w), lw=0.7, **style[kind])
            if kind == "cat":
                ax2.plot(sign*np.real(w.mean()), sign*np.imag(w.mean()), "o", ms=1.5, color=style[kind]["color"], alpha=1)
    for sign in signs:
        ax2.plot(sign*np.real(sp), sign*np.imag(sp), color="gray", lw=0.4)
    ax2.set_xlim(-lims[which], lims[which]); ax2.set_ylim(-lims[which], lims[which]); ax2.set_aspect("equal")
    ax2.set_xlabel(rf"Re ${lab}_R$"); ax2.set_ylabel(rf"Im ${lab}_R$")
handles = [Line2D([], [], lw=1, **s) for s in style.values()] + [Line2D([], [], color="gray", lw=0.5)]
fig.legend(handles, list(style) + ["orbit centres along the branch"], fontsize=5, loc="lower center", ncol=2, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(FIG_DIR + "limit_cycles_branch_3d_20_10_2026.pdf")
plt.show()

## Answer to point 9
- **The cat orbit never loses stability while it exists.** Along the whole branch, at every $\varepsilon_d/g_2$ in the map, the multipliers stay at $|\mu| \le 0.973$.
  They form two complex pairs whose modulus is fixed by Liouville, $\prod|\mu| = e^{-2\pi\,\mathrm{tr}(M^{-1}C)}$. There is no period doubling and no Neimark–Sacker on the cat.
- **The cat disappears through a fold (saddle-node of limit cycles).** At $\varepsilon_d = 10.12\,g_2$ this happens at $g_2^\ast/2\pi \simeq 97$ MHz. Just before it, the slow complex pair collides on the real axis.
  One multiplier then leaves the unit circle through $+1$ exactly at the fold, and the branch continues backwards as a saddle orbit, with $\mu \approx 1.3$, larger amplitude, and unstable.
  Above $g_2^\ast$ the cat is gone. Every start then ends up in the chaotic set or on another attractor.
  This is why the Floquet sweep cannot be pushed to high $\varepsilon_p$: there is no cat solution left to compute.
- **The fold is analytical.** It follows from the article's reduced harmonic balance on the cat family ($\sigma = -1$):
  $g_2^\ast|\alpha| = \sqrt{s^\ast}\,\omega_a\varphi_a\varphi_b/(2\varphi_a^2+\varphi_b^2)$, with $s^\ast = 1/56$ at leading order and $(1+30\rho/49)/56$ with the $3\omega_a$ sideband.
  This gives $g_2^\ast|\alpha|/2\pi \approx 216$ MHz, against 219 MHz from Hill. It is almost independent of $\kappa_b$. The reduced system with the full cosine reproduces the Hill line to 0.1 MHz.
- **The cat-start chaos onset lies 6–9% (5.5–11 MHz) below the fold, at every $\varepsilon_d/g_2$ where both exist.** The two lines are parallel. In this window the cat is still stable,
  but its basin shrinks as the fold approaches, and the initial state $a = i|\alpha|$, $b = 0$ leaves it. The empty buffer is the larger part of the mismatch.
  So the "cat start" boundary of the map is a basin boundary, a coexistence of the cat with chaos, and not an instability of the cat.
  The true loss of the cat is the fold line.